<a href="https://colab.research.google.com/github/dasco572/Projet-ML/blob/main/Projet_25_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
donnees = pd.read_csv("scitweets_export.tsv", sep='\t')

Afficher des lignes pour voir comment l'info est

In [ ]:
donnees.head()

,Unnamed: 0,tweet_id,text,science_related,scientific_claim,scientific_reference,scientific_context
0,0,316669998137483264,Knees are a bit sore. i guess that's a sign th...,0,0.0,0.0,0.0
1,1,319090866545385472,McDonald's breakfast stop then the gym 🏀💪,0,0.0,0.0,0.0
2,2,322030931022065664,Can any Gynecologist with Cancer Experience ex...,1,1.0,0.0,0.0
3,3,322694830620807168,Couch-lock highs lead to sleeping in the couch...,1,1.0,0.0,0.0
4,4,328524426658328576,Does daily routine help prevent problems with ...,1,1.0,0.0,0.0


Pour analyser les tweets il faut qu'on procéde au Nettoyage du texte (pré-traitement), c'est a dire que les textes bruts contiennent souvent des éléments inutiles (liens, ponctuation, emojis…) qui peuvent perturber l’analyse.

cependant on va faire une fonction pour nettoyer cela : elle prend un tweet brut en entrée et renvoie un texte brut analysable par le system .

Pour cela, plusieurs opérations sont effectuées :

- **Uniformiser le texte** : on convertit tout en minuscules pour que les mots identiques majuscule ou minuscule soit pareil pour l'algo

- **Supprimer les éléments inutiles** : on enlève les liens internet qui n’apportent pas d’infos pour l’analyse du contenu.

- **Nettoyer la ponctuation** : on retire les caractères spéciaux et la ponctuation complexe pour ne garder que les mots importants.

- **Nettoyer les emojis** : pour simplifier, on choisit ici de supprimer les emojis.

À la fin de cette étape, les tweets sont plus homogènes, plus lisibles et surtout plus faciles à traiter pour les modèles de machine learning.

In [ ]:

import re

def nettoyer_texte(texte):
    # Sécurité : si la case est vide ou n'est pas du texte, on renvoie du vide
    if type(texte) is not str:
        return ""

    # tranformation en minuscule
    texte = texte.lower()

    # recherche les liens et les retire
    texte = re.sub(r"http\S+", "", texte)

    # retirer les mention
    texte = re.sub(r"@\w+", "", texte)

    # remplacement de tout ce qui n'est pas un texte par un vide
    texte = re.sub(r"[^a-z0-9\s]", "", texte)

    # renitialise les espaces a la fin
    texte = " ".join(texte.split())

    return texte

In [ ]:
# attibuer la fonction sur notre fichier
donnees['texte_propre'] = donnees['text'].apply(nettoyer_texte)

donnees['text'].head()

,text
0,Knees are a bit sore. i guess that's a sign th...
1,McDonald's breakfast stop then the gym 🏀💪
2,Can any Gynecologist with Cancer Experience ex...
3,Couch-lock highs lead to sleeping in the couch...
4,Does daily routine help prevent problems with ...


In [ ]:
donnees['texte_propre'].head()

,texte_propre
0,knees are a bit sore i guess thats a sign that...
1,mcdonalds breakfast stop then the gym
2,can any gynecologist with cancer experience ex...
3,couchlock highs lead to sleeping in the couch ...
4,does daily routine help prevent problems with ...


In [ ]:
# import la boîte à outils pour le langage NLTK
import nltk
from nltk.corpus import stopwords
# telechargement dictionnaire des mots vides et chargement en anglais
nltk.download('stopwords')
mots_vides_anglais = stopwords.words('english')

# On crée notre nouveau filtre
def enlever_mots_vides(texte):
    mots = texte.split()
    mots_utiles = [mot for mot in mots if mot not in mots_vides_anglais]
    return " ".join(mots_utiles)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
donnees['texte_final'] = donnees['texte_propre'].apply(enlever_mots_vides)
donnees[['texte_propre', 'texte_final']].head()

,texte_propre,texte_final
0,knees are a bit sore i guess thats a sign that...,knees bit sore guess thats sign recent treadmi...
1,mcdonalds breakfast stop then the gym,mcdonalds breakfast stop gym
2,can any gynecologist with cancer experience ex...,gynecologist cancer experience explain dangers...
3,couchlock highs lead to sleeping in the couch ...,couchlock highs lead sleeping couch gotta stop...
4,does daily routine help prevent problems with ...,daily routine help prevent problems bipolar di...


In [13]:
donnees.shape

(1140, 9)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

traducteur = TfidfVectorizer(max_features=1000)

# 3attribuer la colonne 'texte_final' pour qu'il apprenne le vocabulaire et donne à nos textes en un grand tableau de nombres
X = traducteur.fit_transform(donnees['texte_final'])

print("taille du tableau de nombres :", X.shape)

taille du tableau de nombres : (1140, 1000)


In [15]:
# import de l'outil pour couper les données en deux
from sklearn.model_selection import train_test_split

y = donnees['science_related']

In [17]:
#decoupage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("essai:", X_train.shape[0])
print("examen :", X_test.shape[0])

essai: 912
examen : 228


Pour faire le tri nous allons commencer par le Naïve Bayes

In [18]:
from sklearn.naive_bayes import MultinomialNB

modele_nb = MultinomialNB()
#entrainement
modele_nb.fit(X_train, y_train)
#test
predictions = modele_nb.predict(X_test)

print("reponses machine :", predictions[:5])
print("vraies reponses: ", y_test[:5].values)

reponses machine : [0 0 0 0 1]
vraies reponses:  [0 0 0 1 1]


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

# bulletin de notes détaillé (Précision, Rappel, F-mesure)
print(" BULLETIN DE NOTES DE LA MACHINE")
# On compare toutes les vraies réponses (y_test) avec toutes les prédictions
print(classification_report(y_test, predictions))

# 3. On affiche la matrice de confusion (le tableau croisé)
print("MATRICE DE CONFUSION ")
print(confusion_matrix(y_test, predictions))

 BULLETIN DE NOTES DE LA MACHINE
              precision    recall  f1-score   support

           0       0.77      0.97      0.85       146
           1       0.89      0.48      0.62        82

    accuracy                           0.79       228
   macro avg       0.83      0.72      0.74       228
weighted avg       0.81      0.79      0.77       228

MATRICE DE CONFUSION 
[[141   5]
 [ 43  39]]


1. Le problème : un modèle trompeur

Notre modèle affiche 79 % de précision, ce qui semble bon.
Mais en réalité, il détecte mal les tweets Scientifiques que à 48 %seulement car les données sont déséquilibrées : il y a beaucoup plus de “Non-Science”.
Du coup, le modèle choisit souvent la réponse la plus facile (“Non-Science”) et rate beaucoup de vrais cas scientifiques.

Pour corriger ça, on fait deux choses :

Utiliser un Pipeline :
On regroupe toutes les étapes (traitement + modèle) dans une seule chaîne pour éviter les erreurs et rendre le processus plus propre.

Équilibrer le modèle :
On utilise une régression logistique avec class_weight='balanced'.
Cela oblige le modèle à faire plus attention aux tweets “Science”, au lieu de les ignorer.

In [20]:
# 1. On importe nos nouveaux outils (la chaîne de montage et la nouvelle machine)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_texte = donnees['texte_final']
y = donnees['science_related']

#  découpage 80% / 20%
X_train_texte, X_test_texte, y_train, y_test = train_test_split(X_texte, y, test_size=0.2, random_state=42)

#  Pipeline
chaine_de_montage = Pipeline([
    ('traducteur', TfidfVectorizer(max_features=1000)),
    ('machine_de_tri', LogisticRegression(class_weight='balanced', random_state=42))
])

# entrainement
chaine_de_montage.fit(X_train_texte, y_train)

# test
nouvelles_predictions = chaine_de_montage.predict(X_test_texte)

print("new bulletin de notes en Pipeline Équilibré")
print(classification_report(y_test, nouvelles_predictions))

print(" MATRICE DE CONFUSION 2")
print(confusion_matrix(y_test, nouvelles_predictions))

new bulletin de notes en Pipeline Équilibré
              precision    recall  f1-score   support

           0       0.85      0.84      0.84       146
           1       0.72      0.74      0.73        82

    accuracy                           0.80       228
   macro avg       0.79      0.79      0.79       228
weighted avg       0.80      0.80      0.80       228

 MATRICE DE CONFUSION 2
[[122  24]
 [ 21  61]]


Préparation des données pour la Tâche 2 : Le Sous-Tri Scientifique

In [21]:
donnees_science = donnees[donnees['science_related'] == 1].copy()
print("Nombre de courriers 100% scientifiques dans notre sac :", len(donnees_science))

# règle du jeu pour la Tâche 2
donnees_science['cible_tache2'] = (donnees_science['scientific_claim'] == 1.0) | (donnees_science['scientific_reference'] == 1.0)

# On transforme le résultat (qui est Vrai/Faux) en 1 et 0 pour notre machine mathématique
donnees_science['cible_tache2'] = donnees_science['cible_tache2'].astype(int)

print("\nRépartition de notre nouveau tri (1 = CLAIM/REF, 0 = CONTEXT) :")
print(donnees_science['cible_tache2'].value_counts())

Nombre de courriers 100% scientifiques dans notre sac : 375

Répartition de notre nouveau tri (1 = CLAIM/REF, 0 = CONTEXT) :
cible_tache2
1    342
0     33
Name: count, dtype: int64


Un grand desequilibre dans le tri
On va prendre en compte le faite qu'il prend en compte plus les erreur faites par le bac context

In [22]:
X_tache2 = donnees_science['texte_final']
y_tache2 = donnees_science['cible_tache2']

X_train_t2, X_test_t2, y_train_t2, y_test_t2 = train_test_split(
    X_tache2, y_tache2, test_size=0.2, random_state=42
)

chaine_de_montage.fit(X_train_t2, y_train_t2)

predictions_t2 = chaine_de_montage.predict(X_test_t2)

print("bulletin de notes tache 2")
print(classification_report(y_test_t2, predictions_t2))

print("matrice tache 2")
print(confusion_matrix(y_test_t2, predictions_t2))

bulletin de notes tache 2
              precision    recall  f1-score   support

           0       1.00      0.29      0.44         7
           1       0.93      1.00      0.96        68

    accuracy                           0.93        75
   macro avg       0.97      0.64      0.70        75
weighted avg       0.94      0.93      0.92        75

matrice tache 2
[[ 2  5]
 [ 0 68]]


Analyse des résultats de la Tâche 2

Le constat : L'illusion des 93 %
En apparence, notre modèle obtient un score global d'Accuracy très élevé (93 %). Cependant, la matrice de confusion et le rapport détaillé révèlent un échec important sur la classe CONTEXT : le Rappel (Recall) n'est que de 0.29 (seuls 2 tweets sur 7 ont été correctement identifiés lors du test). À l'inverse, le modèle réalise un sans-faute sur la classe majoritaire CLAIM/REF (Rappel de 1.00).

Cet échec n'est pas dû à un mauvais paramétrage informatique, mais à un manque cruel de données. Notre sac de courriers d'entraînement contient 342 exemples d'affirmations/références, contre seulement 33 exemples de contexte.
 même avec l'option class_weight='balanced' qui force l'algorithme à être plus attentif, le traducteur vectoriel (TF-IDF) n'a pas pu lire assez d'exemples de la classe CONTEXT pour en dégager un vocabulaire spécifique récurrent. Face au doute et au manque de repères, la machine parie logiquement sur le format qu'elle connaît le mieux.

Tache 3

In [23]:
def priorite_tache3(lettre):
    if lettre['scientific_reference'] == 1.0:
        return 'REF'
    elif lettre['scientific_claim'] == 1.0:
        return 'CLAIM'
    else:
        return 'CONTEXT'

# 2. On applique cette règle à toutes nos lettres scientifiques pour créer nos 3 étiquettes
donnees_science['cible_tache3'] = donnees_science.apply(priorite_tache3, axis=1)

# 3. On regarde combien de lettres se retrouvent dans chaque bac
print(" RÉPARTITION DE NOS 3 BACS ")
print(donnees_science['cible_tache3'].value_counts())

 RÉPARTITION DE NOS 3 BACS 
cible_tache3
REF        203
CLAIM      139
CONTEXT     33
Name: count, dtype: int64


In [24]:
# 1. On prépare la matière première de notre Tâche 3
X_tache3 = donnees_science['texte_final']
y_tache3 = donnees_science['cible_tache3']

X_train_t3, X_test_t3, y_train_t3, y_test_t3 = train_test_split(
    X_tache3, y_tache3, test_size=0.2, random_state=42
)

chaine_de_montage.fit(X_train_t3, y_train_t3)

predictions_t3 = chaine_de_montage.predict(X_test_t3)
print("BULLETIN TACHE 3 ")
print(classification_report(y_test_t3, predictions_t3))

print("MATRICE DE CONFUSION TACHE 3 ")

import pandas as pd
matrice_t3 = confusion_matrix(y_test_t3, predictions_t3, labels=['CLAIM', 'CONTEXT', 'REF'])
print(pd.DataFrame(matrice_t3, index=['Vrai CLAIM', 'Vrai CONTEXT', 'Vrai REF'], columns=['Prédit CLAIM', 'Prédit CONTEXT', 'Prédit REF']))

BULLETIN TACHE 3 
              precision    recall  f1-score   support

       CLAIM       0.56      0.52      0.54        29
     CONTEXT       1.00      0.29      0.44         7
         REF       0.59      0.69      0.64        39

    accuracy                           0.59        75
   macro avg       0.71      0.50      0.54        75
weighted avg       0.61      0.59      0.58        75

MATRICE DE CONFUSION TACHE 3 
              Prédit CLAIM  Prédit CONTEXT  Prédit REF
Vrai CLAIM              15               0          14
Vrai CONTEXT             0               2           5
Vrai REF                12               0          27


Analyse des résultats de la Tâche 3 : La complexité du Multi-label

1. Le constat : La confusion entre Affirmation et Référence
Pour cette tâche à trois classes, l'Accuracy globale chute à 59 %, avec un Macro F1-score de 0.54. L'analyse de la matrice de confusion montre que l'erreur principale de l'algorithme ne vient pas du hasard, mais d'une confusion symétrique forte entre les classes CLAIM et REF. Près de la moitié des véritables assertions (14/29) sont prédites comme des références, et un tiers des véritables références (12/39) sont prédites comme des assertions. La classe minoritaire CONTEXT reste quant à elle très mal identifiée à cause de sa rareté.

2. L'explication logique : La frontière sémantique
Cette baisse de performance illustre parfaitement la difficulté inhérente à la nature "multi-labels" du jeu de données SciTweets. Dans la réalité du réseau social, un tweet contient très souvent simultanément une affirmation scientifique et le lien (la référence) qui la soutient. Le vocabulaire (les variables TF-IDF) des deux classes se chevauche donc énormément. En imposant une règle de classification stricte (un tweet = une seule classe), nous forçons le modèle à trancher sur des frontières sémantiques qui sont naturellement poreuses et ambiguës, ce qui génère inévitablement de nombreuses erreurs de prédiction.

Analyse d'erreur qualitative comme demandé dans notre plan

In [30]:
resultats = pd.DataFrame({
    'Texte_Nettoye': X_test_t3, # Le texte que la machine a lu
    'Vraie_Reponse': y_test_t3, # Le bac correct
    'Prediction_Machine': predictions_t3 # Le bac choisi par l'algorithme
})

# 2. Le Filtre Magique : On ne garde QUE les lettres où la prédiction est différente (!=) de la vraie réponse
erreurs = resultats[resultats['Vraie_Reponse'] != resultats['Prediction_Machine']]

print("5 courriers mal triés")
for index, ligne in erreurs.head(5).iterrows():
    print(f"TEXTE LU : {ligne['Texte_Nettoye']}")
    print(f"VRAI BAC : {ligne['Vraie_Reponse']}   ERREUR DE LA MACHINE : {ligne['Prediction_Machine']}")
    print(" " * 10)

5 courriers mal triés
TEXTE LU : rt bbcsciencenews database helps plant right tree right place
VRAI BAC : REF   ERREUR DE LA MACHINE : CLAIM
          
TEXTE LU : know eat affect productivity level
VRAI BAC : REF   ERREUR DE LA MACHINE : CLAIM
          
TEXTE LU : biotechnology efficient stem cells grow animals
VRAI BAC : CLAIM   ERREUR DE LA MACHINE : REF
          
TEXTE LU : study marijuana decriminalization leads decreased arrests increase youth use
VRAI BAC : REF   ERREUR DE LA MACHINE : CLAIM
          
TEXTE LU : 5 surprising ways social media affecting healthcare social media frontiers
VRAI BAC : CLAIM   ERREUR DE LA MACHINE : REF
          


Analyse Qualitative des Erreurs (Tâche 3)

L'inspection manuelle des tweets mal classés par notre modèle révèle deux causes principales d'erreurs, qui justifient les scores obtenus et mettent en lumière les limites de notre approche :

1. La perte de signal due au pré-traitement (Suppression des URLs)
Plusieurs tweets classés à tort comme des affirmations (CLAIM) étaient en réalité des références (REF). Exemple : "know eat affect productivity level". La lecture du texte brut nettoyé montre qu'il ne reste qu'une affirmation. En supprimant les URLs (les liens http) lors de notre étape d'ingénierie des données textuelles, nous avons effacé le marqueur le plus fort de la classe REF. Le modèle n'avait donc plus les moyens de deviner qu'il s'agissait d'un lien vers un article.

2. L'ambiguïté sémantique (Le paradoxe de la conclusion d'étude)
Le modèle échoue souvent face à des tweets qui formulent l'affirmation d'une étude. Exemple : "study marijuana decriminalization leads decreased arrests...". Bien que le mot "study" indique une référence (REF, la véritable étiquette), la majorité du vocabulaire du tweet est constituée d'une assertion forte. L'algorithme (TF-IDF) donne un poids mathématique supérieur aux termes de l'affirmation et classe le tweet en CLAIM. Cela confirme la difficulté d'imposer une classification unique (mutuellement exclusive) sur un jeu de données intrinsèquement multi-labels où la référence sert souvent de support direct à l'affirmation.